In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.append(str(REPO_ROOT))

import torch
import numpy as np
import matplotlib.pyplot as plt
import time
from collections import Counter, defaultdict

from causallearn.search.ConstraintBased.PC import pc
from causallearn.search.ScoreBased.GES import ges
from causallearn.utils.cit import chisq

DATA_DIR    = REPO_ROOT / "data"
FIGURES_DIR = REPO_ROOT / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

In [2]:
# Load the PC-learned graph + binarized matrix from Day 3
graph_data = torch.load(DATA_DIR / "pc_learned_graph.pt", weights_only=False)
edges_orig    = graph_data["edges"]
col_to_layer  = graph_data["col_to_layer"]
col_to_feat   = graph_data["col_to_feat"]
LAYERS        = graph_data["layers"]
X_binary      = graph_data["X_binary"]    # [N, 115] used to train PC originally
n_features = len(col_to_layer)
N = X_binary.shape[0]

# Load the case study so we can track its persistence
case = torch.load(DATA_DIR / "case_study_and_validation.pt", weights_only=False)
A_case = case["case_study"]["A"]["col"]
B_case = case["case_study"]["B"]["col"]
C_case = case["case_study"]["C"]["col"]
print(f"Loaded:")
print(f"  X_binary shape: {X_binary.shape}")
print(f"  Original edges: {len(edges_orig)}")
print(f"  Case study v-structure: cols ({A_case}, {C_case}, {B_case})")
print(f"    A: L{col_to_layer[A_case]} F{col_to_feat[A_case]}")
print(f"    B: L{col_to_layer[B_case]} F{col_to_feat[B_case]}")
print(f"    C: L{col_to_layer[C_case]} F{col_to_feat[C_case]}")


Loaded:
  X_binary shape: (5000, 115)
  Original edges: 366
  Case study v-structure: cols (13, 60, 27)
    A: L18 F15143
    B: L18 F2297
    C: L22 F10566


In [3]:
# Helper: extract edges and v-structures from a causal-learn CausalGraph output
def extract_edges_and_vstructs(G_matrix, col_to_layer):
    """
    Convert a causal-learn output to:
      - edges: set of (parent_col, child_col) — directed by layer prior for cross-layer,
               by PC's orientation otherwise; undirected stored as frozenset
      - vstructs: list of (A, C, B) tuples where A → C ← B and A,B not adjacent
    """
    n = G_matrix.shape[0]
    directed_edges = set()
    undirected_edges = set()
    adjacency = {i: set() for i in range(n)}
    parents = {i: set() for i in range(n)}

    for i in range(n):
        for j in range(i + 1, n):
            edge_present = (G_matrix[i, j] != 0) or (G_matrix[j, i] != 0)
            if not edge_present:
                continue
            adjacency[i].add(j)
            adjacency[j].add(i)
            L_i = col_to_layer[i] if col_to_layer else None
            L_j = col_to_layer[j] if col_to_layer else None

            # Cross-layer: orient by layer prior
            if col_to_layer and L_i != L_j:
                if L_i < L_j:
                    directed_edges.add((i, j))
                    parents[j].add(i)
                else:
                    directed_edges.add((j, i))
                    parents[i].add(j)
            else:
                # Use PC's own orientation if available
                if G_matrix[i, j] == -1 and G_matrix[j, i] == 1:
                    directed_edges.add((i, j))
                    parents[j].add(i)
                elif G_matrix[i, j] == 1 and G_matrix[j, i] == -1:
                    directed_edges.add((j, i))
                    parents[i].add(j)
                else:
                    undirected_edges.add(frozenset([i, j]))

    # Find v-structures from parent sets
    vstructs = []
    for c in range(n):
        plist = sorted(parents[c])
        for i, A in enumerate(plist):
            for B in plist[i+1:]:
                if B not in adjacency[A]:
                    vstructs.append((A, c, B))

    return directed_edges, undirected_edges, vstructs

# Reconstruct the original (Day 3) edge set in the same format
edges_orig_set = set()
for p, c, _ in edges_orig:
    edges_orig_set.add((p, c))

print(f"Reconstructed {len(edges_orig_set)} edges from Day 3 graph for comparison.")


Reconstructed 366 edges from Day 3 graph for comparison.


In [4]:
import numpy as np

print(f"X_binary type: {type(X_binary)}")
print(f"X_binary dtype: {X_binary.dtype}")
print(f"X_binary shape: {X_binary.shape}")
if hasattr(X_binary, 'device'):
    print(f"X_binary device: {X_binary.device}")
print(f"X_binary[0, :5]: {X_binary[0, :5]}")

# Try a tiny PC run to see if it works at all
print("\nTrying tiny PC test (5 features, 100 samples)...")
import time
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import chisq

X_small = X_binary[:100, :5]
print(f"X_small type: {type(X_small)}, dtype: {X_small.dtype}")

t0 = time.time()
try:
    cg_test = pc(data=X_small, alpha=0.05, indep_test=chisq, show_progress=False)
    print(f"PC tiny run completed in {time.time() - t0:.2f}s")
except Exception as e:
    print(f"PC failed: {type(e).__name__}: {e}")

X_binary type: <class 'numpy.ndarray'>
X_binary dtype: int8
X_binary shape: (5000, 115)
X_binary device: cpu
X_binary[0, :5]: [0 1 0 1 1]

Trying tiny PC test (5 features, 100 samples)...
X_small type: <class 'numpy.ndarray'>, dtype: int8
PC tiny run completed in 0.00s


In [ ]:
from tqdm import tqdm

# Run from stricter to looser alpha. Later values can be much slower.
ALPHAS = [0.001, 0.005, 0.01, 0.05, 0.1]
alpha_results = []

# stable=True is the standard PC variant (more conservative, often slower).
# Set to False for faster exploratory runs.
PC_STABLE = True

print(f"Running PC at {len(ALPHAS)} alpha values sequentially...")
print("(α=0.001 is usually fastest; α=0.05 and 0.1 may take much longer)")
print(f"PC stable mode: {PC_STABLE}")
print()

for alpha in tqdm(ALPHAS, desc="Alpha sweep"):
    print(f"Starting alpha={alpha:.3f}...", flush=True)
    t0 = time.time()
    cg = pc(
        data=X_binary,
        alpha=alpha,
        indep_test=chisq,
        stable=PC_STABLE,
        show_progress=True,
    )
    elapsed = time.time() - t0

    G = cg.G.graph
    directed, undirected, vstructs = extract_edges_and_vstructs(G, col_to_layer)
    total_edges = len(directed) + len(undirected)
    cross_layer_edges = sum(1 for p, c in directed
                            if col_to_layer[p] != col_to_layer[c])

    case_present = any(
        (A == A_case and C == C_case and B == B_case) or
        (A == B_case and C == C_case and B == A_case)
        for A, C, B in vstructs
    )

    alpha_results.append({
        "alpha":             alpha,
        "total_edges":       total_edges,
        "cross_layer_edges": cross_layer_edges,
        "n_vstructs":        len(vstructs),
        "case_present":      case_present,
        "runtime_s":         elapsed,
        "stable":            PC_STABLE,
    })
    tqdm.write(f"  α={alpha:.3f}: {total_edges:>4d} edges ({cross_layer_edges} cross-layer), "
               f"{len(vstructs):>4d} v-structs, "
               f"case={'✓' if case_present else '✗'}, "
               f"{elapsed:.1f}s")

Running PC at 5 alpha values sequentially...
(α=0.001 is usually fastest; α=0.05 and 0.1 may take much longer)
PC stable mode: True



Alpha sweep:   0%|          | 0/5 [00:00<?, ?it/s]

Starting alpha=0.001...


  0%|          | 0/115 [00:00<?, ?it/s]

In [ ]:
# Summary table
print("=== Alpha Sweep Summary ===")
print(f"{'alpha':>8s}  {'edges':>6s}  {'cross-layer':>12s}  {'v-structs':>10s}  {'case?':>6s}")
for r in alpha_results:
    print(f"{r['alpha']:>8.3f}  {r['total_edges']:>6d}  {r['cross_layer_edges']:>12d}  "
          f"{r['n_vstructs']:>10d}  {str(r['case_present']):>6s}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
alphas_arr   = [r["alpha"] for r in alpha_results]
edges_arr    = [r["total_edges"] for r in alpha_results]
vstructs_arr = [r["n_vstructs"] for r in alpha_results]

axes[0].semilogx(alphas_arr, edges_arr, "o-", label="Total edges")
axes[0].semilogx(alphas_arr, [r["cross_layer_edges"] for r in alpha_results],
                 "s-", label="Cross-layer edges")
axes[0].axvline(0.01, color="grey", linestyle=":", label="α used in main result")
axes[0].set_xlabel("alpha"); axes[0].set_ylabel("edge count"); axes[0].legend()
axes[0].set_title("Edges vs. α")
axes[0].grid(True, alpha=0.3)

axes[1].semilogx(alphas_arr, vstructs_arr, "o-", color="green")
axes[1].axvline(0.01, color="grey", linestyle=":")
axes[1].set_xlabel("alpha"); axes[1].set_ylabel("v-structure count")
axes[1].set_title("V-structures vs. α")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "alpha_sweep.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
import multiprocessing as mp
from functools import partial
from tqdm.auto import tqdm

N_BOOTSTRAP = 50
ALPHA_BOOT = 0.01

# Detect available cores. Cap at 16 to avoid PC's chi-squared
# bottlenecking on shared cache lines.
N_WORKERS = min(mp.cpu_count() - 1, 16)
print(f"Using {N_WORKERS} parallel workers (system has {mp.cpu_count()} cores)")
print(f"Running {N_BOOTSTRAP} bootstrap iterations...")


def bootstrap_iteration(seed, X_binary_arr, col_to_layer_list, alpha, A_case, B_case, C_case):
    """
    One bootstrap iteration. Resamples, runs PC, returns edges/vstructs/case_present.
    Standalone so it can be pickled for multiprocessing.
    """
    import numpy as np
    from causallearn.search.ConstraintBased.PC import pc
    from causallearn.utils.cit import chisq

    rng = np.random.RandomState(seed)
    N = X_binary_arr.shape[0]
    indices = rng.choice(N, size=N, replace=True)
    X_boot = X_binary_arr[indices]

    nonconst = X_boot.var(axis=0) > 0
    X_boot_sub = X_boot[:, nonconst]
    col_map = np.where(nonconst)[0]

    if not nonconst.all():
        boot_col_to_layer = [col_to_layer_list[col_map[c]] for c in range(X_boot_sub.shape[1])]
    else:
        boot_col_to_layer = list(col_to_layer_list)

    try:
        cg = pc(data=X_boot_sub, alpha=alpha, indep_test=chisq, show_progress=False)
        G = cg.G.graph
        n = G.shape[0]

        # Inline edge/v-struct extraction (can't import the helper through pickle reliably)
        directed_edges = set()
        undirected_edges = set()
        adjacency = {i: set() for i in range(n)}
        parents = {i: set() for i in range(n)}

        for i in range(n):
            for j in range(i + 1, n):
                if (G[i, j] == 0) and (G[j, i] == 0):
                    continue
                adjacency[i].add(j)
                adjacency[j].add(i)
                L_i, L_j = boot_col_to_layer[i], boot_col_to_layer[j]
                if L_i != L_j:
                    if L_i < L_j:
                        directed_edges.add((i, j))
                        parents[j].add(i)
                    else:
                        directed_edges.add((j, i))
                        parents[i].add(j)
                else:
                    if G[i, j] == -1 and G[j, i] == 1:
                        directed_edges.add((i, j))
                        parents[j].add(i)
                    elif G[i, j] == 1 and G[j, i] == -1:
                        directed_edges.add((j, i))
                        parents[i].add(j)
                    else:
                        undirected_edges.add(frozenset([i, j]))

        vstructs = []
        for c in range(n):
            plist = sorted(parents[c])
            for i, A in enumerate(plist):
                for B in plist[i+1:]:
                    if B not in adjacency[A]:
                        vstructs.append((A, c, B))

        # Map back to original column indices
        edges_orig_indexed = []
        for p, c in directed_edges:
            edges_orig_indexed.append((int(col_map[p]), int(col_map[c])))
        for fs in undirected_edges:
            i, j = list(fs)
            edges_orig_indexed.append((int(col_map[i]), int(col_map[j])))

        local_vstructs = set()
        for A, C, B in vstructs:
            tup_a = int(col_map[A])
            tup_c = int(col_map[C])
            tup_b = int(col_map[B])
            local_vstructs.add(tuple(sorted([tup_a, tup_b]) + [tup_c]))

        case_key = tuple(sorted([A_case, B_case]) + [C_case])
        case_present_local = case_key in local_vstructs

        return {
            "edges":            edges_orig_indexed,
            "vstructs":         list(local_vstructs),
            "case_present":     case_present_local,
            "success":          True,
        }
    except Exception as e:
        return {"success": False, "error": str(e)}


# Run bootstraps in parallel
seeds = list(range(N_BOOTSTRAP))   # deterministic seeds 0..N_BOOTSTRAP-1
worker_fn = partial(
    bootstrap_iteration,
    X_binary_arr=X_binary,
    col_to_layer_list=list(col_to_layer),
    alpha=ALPHA_BOOT,
    A_case=A_case,
    B_case=B_case,
    C_case=C_case,
)

t0 = time.time()
edge_counts = Counter()
vstruct_counts = Counter()
case_persist = 0
n_failed = 0

with mp.Pool(N_WORKERS) as pool:
    pbar = tqdm(total=N_BOOTSTRAP, desc="Bootstrapping")
    for result in pool.imap_unordered(worker_fn, seeds):
        if not result["success"]:
            n_failed += 1
            tqdm.write(f"  PC failed: {result.get('error', 'unknown')}")
            pbar.update(1)
            continue

        for e in result["edges"]:
            edge_counts[e] += 1
        for vs in result["vstructs"]:
            vstruct_counts[vs] += 1
        if result["case_present"]:
            case_persist += 1
        pbar.set_postfix(case=f"{case_persist}/{pbar.n+1}")
        pbar.update(1)
    pbar.close()

elapsed = time.time() - t0
print(f"\nBootstrap complete in {elapsed/60:.1f} min "
      f"({elapsed/N_BOOTSTRAP:.1f}s per iteration on average)")
print(f"  Successful: {N_BOOTSTRAP - n_failed}/{N_BOOTSTRAP}")
print(f"  Failed: {n_failed}")
print(f"  Case study v-structure persistence: "
      f"{case_persist}/{N_BOOTSTRAP - n_failed} "
      f"({100*case_persist/max(1, N_BOOTSTRAP - n_failed):.0f}%)")


In [ ]:
# Edge stability distribution
print("=== Bootstrap Edge Stability ===")
n_edges_total = len(edge_counts)
high_stab = sum(1 for c in edge_counts.values() if c >= 0.8 * N_BOOTSTRAP)
mid_stab  = sum(1 for c in edge_counts.values() if 0.5 * N_BOOTSTRAP <= c < 0.8 * N_BOOTSTRAP)
low_stab  = sum(1 for c in edge_counts.values() if c < 0.5 * N_BOOTSTRAP)
print(f"  Total distinct edges seen across bootstraps: {n_edges_total}")
print(f"  High stability (≥80%): {high_stab}")
print(f"  Medium stability (50-80%): {mid_stab}")
print(f"  Low stability (<50%): {low_stab}")

# How many edges from the original Day 3 graph appear in ≥80% of bootstraps?
orig_high_stab = sum(1 for e in edges_orig_set
                     if edge_counts.get(e, 0) >= 0.8 * N_BOOTSTRAP)
print(f"\n  Of original {len(edges_orig_set)} edges, "
      f"{orig_high_stab} appear in ≥80% of bootstraps "
      f"({100*orig_high_stab/len(edges_orig_set):.0f}%)")

# Histogram
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist([c / N_BOOTSTRAP for c in edge_counts.values()], bins=20, edgecolor="black")
ax.axvline(0.8, color="red", linestyle="--", label="High-stability threshold (80%)")
ax.set_xlabel("Bootstrap appearance frequency")
ax.set_ylabel("Number of edges")
ax.set_title("Bootstrap stability of PC's edges")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "bootstrap_stability.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
from tqdm.auto import tqdm

print("Running GES with BIC...")
print("(GES is single-threaded; this may take 3-15 min depending on graph density)")
t0 = time.time()

with tqdm(total=1, desc="GES", bar_format='{desc}: {elapsed} elapsed') as pbar:
    X_for_ges = X_binary.astype(float)
    ges_result = ges(X_for_ges, score_func="local_score_BIC")
    pbar.update(1)

print(f"\nGES finished in {time.time() - t0:.1f}s")

G_ges = ges_result["G"].graph
directed_ges, undirected_ges, vstructs_ges = extract_edges_and_vstructs(G_ges, col_to_layer)
total_ges = len(directed_ges) + len(undirected_ges)
cross_ges = sum(1 for p, c in directed_ges if col_to_layer[p] != col_to_layer[c])

print(f"\nGES output:")
print(f"  Edges: {total_ges} (cross-layer {cross_ges})")
print(f"  V-structures: {len(vstructs_ges)}")

ges_edges_set = directed_ges | {tuple(sorted(fs)) for fs in undirected_ges}

def to_undirected_set(edges):
    return {frozenset([p, c]) for p, c in edges if p != c}

pc_und  = to_undirected_set(edges_orig_set)
ges_und = to_undirected_set(directed_ges) | undirected_ges

intersection = pc_und & ges_und
union = pc_und | ges_und
jaccard = len(intersection) / len(union) if union else 0

print(f"\nPC vs GES edge overlap (undirected):")
print(f"  PC edges:    {len(pc_und)}")
print(f"  GES edges:   {len(ges_und)}")
print(f"  Intersection: {len(intersection)}")
print(f"  Union:        {len(union)}")
print(f"  Jaccard:      {jaccard:.3f}")

case_in_ges = any(
    (A == A_case and C == C_case and B == B_case) or
    (A == B_case and C == C_case and B == A_case)
    for A, C, B in vstructs_ges
)
print(f"\nCase study v-structure in GES output: {case_in_ges}")


In [ ]:
print("=== Case Study Persistence Summary ===")
print()
print("Across alpha values:")
case_alpha_count = sum(1 for r in alpha_results if r["case_present"])
for r in alpha_results:
    print(f"  α = {r['alpha']:.3f}: {'present' if r['case_present'] else 'absent'}")
print(f"  Total: {case_alpha_count}/{len(alpha_results)} alpha values")
print()
print(f"Across {N_BOOTSTRAP} bootstrap iterations:")
print(f"  Present in: {case_persist}/{N_BOOTSTRAP} ({100*case_persist/N_BOOTSTRAP:.0f}%)")
print()
print("In GES (different algorithm):")
print(f"  Present: {case_in_ges}")


In [ ]:
torch.save({
    "alpha_sweep":          alpha_results,
    "bootstrap_edge_counts": dict(edge_counts),
    "bootstrap_vstruct_counts": dict(vstruct_counts),
    "n_bootstrap":          N_BOOTSTRAP,
    "case_persist_bootstrap": case_persist,
    "ges_edges":            list(directed_ges) + [tuple(fs) for fs in undirected_ges],
    "ges_vstructs":         vstructs_ges,
    "case_in_ges":          case_in_ges,
    "pc_ges_jaccard":       jaccard,
}, DATA_DIR / "robustness_results.pt")
print(f"Saved robustness results to {DATA_DIR / 'robustness_results.pt'}")

print("\n=== Day 5 (Robustness) complete ===")
print()
print("Key numbers for the report:")
print(f"  Case study v-structure persists at {case_alpha_count}/{len(alpha_results)} alpha values")
print(f"  Case study v-structure persists in {case_persist}/{N_BOOTSTRAP} bootstrap runs ({100*case_persist/N_BOOTSTRAP:.0f}%)")
print(f"  PC vs GES edge overlap (Jaccard): {jaccard:.3f}")
print(f"  Of original {len(edges_orig_set)} PC edges, {orig_high_stab} ({100*orig_high_stab/len(edges_orig_set):.0f}%) are bootstrap-stable (≥80%)")
